# Hands-on 4

**MNIST Classification using Tensorflow**

In [ ]:
"""
Minimal MNIST classifier using tensorflow
"""

import numpy as np
import tensorflow as tf

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# plot 4 images as gray scale
sp1 = plt.subplot(141)
sp1.axis(False)
sp1.set_title(y_train[0])
sp1.imshow(X_train[0], cmap=plt.get_cmap('gray'))
sp2 = plt.subplot(142)
sp2.axis(False)
sp2.set_title(y_train[1])
sp2.imshow(X_train[1], cmap=plt.get_cmap('gray'))
sp3 = plt.subplot(143)
sp3.axis(False)
sp3.set_title(y_train[2])
sp3.imshow(X_train[2], cmap=plt.get_cmap('gray'))
sp4 = plt.subplot(144)
sp4.axis(False)
sp4.set_title(y_train[3])
sp4.imshow(X_train[3], cmap=plt.get_cmap('gray'))
# show the plot
plt.tight_layout()
plt.show()

# Preprocess: scale to [0,1] and flatten
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
x_train = x_train.reshape((-1, 28 * 28))
x_test  = x_test.reshape((-1, 28 * 28))

# Build model (simple fully connected)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28 * 28,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax")
], name="mnist_dense")

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Show model summary
model.summary()

# Train
EPOCHS = 5
BATCH_SIZE = 32
model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2
)

# Evaluate
loss, acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {acc:.4f}")


**MNIST Classification using Pytorch**

In [ ]:
"""
mnist_pytorch_simple.py
"""

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data
transform = transforms.ToTensor()  # converts to [0,1] and adds channel dim
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256)

# Model: simple MLP
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10)
).to(device)

# Loss and optimizer
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

# Training loop (3 epochs for illustration)
epochs = 3
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    print(f"Epoch {epoch:02d}  Train loss: {avg_loss:.4f}  Train acc: {acc:.4f}")

# Evaluation on test set
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)
        test_correct += (preds == yb).sum().item()
        test_total += xb.size(0)

test_acc = test_correct / test_total
print(f"Test accuracy: {test_acc:.4f}")
